# Cursor Remote SSH + CUDA workbook

This is the complete remote workflow: connect to a Linux/NVIDIA host, prepare a remote checkout, start a remote folder-local kernel, verify CUDA, run PyTorch, and release the GPU.

## 0. Start from an empty folder

If you are creating a new project rather than using this example repository, start on the Mac with:

```bash
mkdir -p ~/Developer/my-remote-ml-project
cd ~/Developer/my-remote-ml-project
uv init --python 3.12
uv add torch numpy matplotlib
uv add --dev jupyter ipykernel pytest ruff
mkdir -p notebooks tests
uv sync
git init
git add .
git commit -m "Initialize notebook project"
cursor -r .
```

Create a notebook in `notebooks/`, select `.venv/bin/python`, run the local smoke test, and push the repository before cloning it on the remote machine. Use this example repository directly if you do not want to create a second repository.

## 1. Provision the GPU box with Terraform (preferred)

Use the existing `anmho/terraform` repository. Its `projects/cursor-gpu` root creates one disposable RunPod NVIDIA pod with direct SSH and outputs the connection details needed by Cursor. Before the first apply, create a RunPod API key and register the public key matching the identity you will use from Cursor. The root intentionally does not manage the account-wide SSH-key set.

```bash
export RUNPOD_API_KEY='YOUR_RUNPOD_API_KEY'
git clone https://github.com/anmho/terraform.git ~/Developer/terraform
cd ~/Developer/terraform
terraform -chdir=projects/cursor-gpu init -reconfigure -backend-config=backend.gcs.hcl
terraform -chdir=projects/cursor-gpu plan
terraform -chdir=projects/cursor-gpu apply
terraform -chdir=projects/cursor-gpu output
```

The default is a community-cloud RTX 3090. Use `ssh_direct_host`, `ssh_direct_port`, and `ssh_direct_username` from the outputs for the SSH entry below. The direct SSH output is the one that supports Cursor Remote SSH and port forwarding. The pod remains billable while running; release it with `terraform -chdir=projects/cursor-gpu destroy` when finished.

## 2. CLI fallback

If Terraform is unavailable, use RunPod's CLI after account/API-key setup:

```bash
runpodctl gpu list
runpodctl pod create \
  --name cursor-notebook-workspace \
  --image "runpod/pytorch:latest" \
  --gpu-id "NVIDIA GeForce RTX 3090" \
  --container-disk-in-gb 20 \
  --ports "22/tcp" \
  --ssh \
  --stop-after 4h
runpodctl pod get POD_ID
```

## 3. Provision the GPU box manually if CLI is unavailable

This notebook cannot create a provider instance without your provider account and billing authorization. In RunPod, Vast, or another GPU provider, manually:

1. Create a Linux GPU instance/Pod with an NVIDIA GPU and a PyTorch/CUDA image.
2. Enable direct SSH and a public IP or reachable SSH endpoint.
3. Add the Mac public key from `~/.ssh/id_ed25519.pub`; never upload the private key.
4. Attach persistent storage for checkpoints or datasets if needed.
5. Set an automatic stop or termination limit.
6. Copy the actual SSH host, port, username, and running status from the provider's connection panel.

For a RunPod Pod, use SSH enabled, expose `22/tcp`, and require a public IP when direct SSH is needed. The Pod details panel supplies the real host and port.

## 4. Before opening Cursor

On the Mac, add that real host to `~/.ssh/config`:

```sshconfig
Host gpu-box
    HostName YOUR_GPU_HOST
    User YOUR_REMOTE_USER
    IdentityFile ~/.ssh/id_ed25519
```

Test SSH and the GPU directly:

```bash
ssh gpu-box nvidia-smi
```

## 5. Prepare the remote checkout

Run these commands on the remote host. Replace the repository URL and folder name if you created a new project:

```bash
mkdir -p ~/Developer
cd ~/Developer
git clone YOUR_REPOSITORY_URL my-remote-ml-project
cd ~/Developer/my-remote-ml-project
uv sync
nvidia-smi
uv run python -c "import torch; available = torch.cuda.is_available(); print({'cuda': available, 'device': torch.cuda.get_device_name(0) if available else None})"
```

If this repository is already checked out remotely, use `git pull` and `uv sync` instead of cloning again.

## 6. Start the remote kernel in Cursor

1. Run `Remote-SSH: Connect to Host...` and choose `gpu-box`.
2. Open the remote project folder you just prepared (for example, `my-remote-ml-project`).
3. Open this notebook from the remote Explorer.
4. In the notebook toolbar, click `Select Kernel` (or the current kernel name).
5. Choose `Select Another Kernel...`.
6. Choose `Python Environments...`, then choose the remote `.venv/bin/python`.
7. Click `Run All`. The first code cell starts the kernel process on the remote machine.

## Keyboard controls on macOS

- Execute the selected cell: `Control+Enter` (`⌃⏎`).
- Add a code cell: `Command+Enter` (`⌘⏎`).
- Execute everything: click `Run All` in the notebook toolbar.

![Cursor notebook kernel picker flow](../docs/images/cursor-kernel-picker-flow.svg)

Do not use the Command Palette's `Jupyter: Manage Access To Jupyter Kernels`. There is no file-sync layer: this notebook and kernel are both remote.

## What success looks like

The first cell prints a remote `.venv` executable, a CUDA version, and an NVIDIA GPU name. The next cells test CUDA matrix multiplication and autograd. This notebook intentionally fails if CUDA is unavailable; it never falls back to CPU or Mac MPS.

## Stop and release resources

Commit or push code, close the notebook, disconnect Remote SSH, and stop or terminate the GPU instance with its provider. Closing Cursor does not stop a rented GPU.

In [ ]:
import platform
import sys

import torch

# This cell runs in the remote Linux kernel selected through Cursor Remote SSH.
# Fail immediately instead of silently running the CUDA notebook on CPU/MPS.
assert torch.cuda.is_available(), "This notebook requires an NVIDIA CUDA runtime."
device = torch.device("cuda")
print(
    {
        "platform": platform.platform(),
        "executable": sys.executable,
        "torch": torch.__version__,
        "cuda": torch.version.cuda,
        "device": torch.cuda.get_device_name(0),
    }
)

In [ ]:
left = torch.ones((1024, 1024), device=device)
right = torch.full((1024, 1024), 2.0, device=device)
product = left @ right
torch.cuda.synchronize()
print(
    {
        "device": str(product.device),
        "shape": tuple(product.shape),
        "sample": float(product[0, 0].cpu()),
    }
)

In [ ]:
weights = torch.randn(1024, 1024, device=device, requires_grad=True)
loss = weights.square().mean()
loss.backward()
torch.cuda.synchronize()
print(
    {"loss": loss.detach().cpu().item(), "gradient_norm": weights.grad.norm().detach().cpu().item()}
)